# Offline policy eval — predicted vs. ground-truth actions

Runs a trained diffusion/RDP policy on a held-out (validation) episode.

Reproduces the val split from the run's cfg, replays one episode through `policy.predict_action` in a sliding receding-horizon loop, then compares the predicted action trajectory against ground truth.

Set `RUN_DIR` (and optionally `CKPT`, `VAL_EPISODE`, `NUM_INFERENCE_STEPS`) in the config cell below.

In [ ]:
from __future__ import annotations

import pathlib
import sys

import dill
import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

%matplotlib inline

# --- config ---------------------------------------------------------------
RUN_DIR             = 'data/outputs/2026.05.11/11.31.38_train_diffusion_unet_image_ee2dice_dp_v1'
CKPT                = None   # str or None — defaults to best top-k by filename loss
VAL_EPISODE         = 0      # 0-indexed position into the validation split
OUT_DIR             = None   # str or None — defaults to <run-dir>/eval/
DEVICE              = 'cuda'
NUM_INFERENCE_STEPS = None   # int or None — overrides policy.num_inference_steps
# --------------------------------------------------------------------------

def _find_project_root() -> pathlib.Path:
    for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
        if (p / 'train.py').is_file() and (p / 'trainflow').is_dir():
            return p
    raise RuntimeError(f'cannot locate project root from {pathlib.Path.cwd()}')

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trainflow.common.replay_buffer import ReplayBuffer
from trainflow.common.sampler import get_val_mask

run_dir = (PROJECT_ROOT / RUN_DIR).resolve() if not pathlib.Path(RUN_DIR).is_absolute() else pathlib.Path(RUN_DIR).resolve()
assert run_dir.is_dir(), f'run_dir not found: {run_dir}'
print(f'project root : {PROJECT_ROOT}')
print(f'run_dir      : {run_dir}')

## 1. Resolve checkpoint & output paths

If `CKPT` is `None`, picks the top-k checkpoint with the lowest loss in its filename, else falls back to `latest.ckpt`.

In [ ]:
def find_best_ckpt(run_dir: pathlib.Path) -> pathlib.Path:
    topk = list((run_dir / 'checkpoints').glob('epoch=*.ckpt'))

    def loss_of(p: pathlib.Path) -> float:
        try:
            return float(p.stem.split('=')[-1])
        except ValueError:
            return float('inf')

    if topk:
        return min(topk, key=loss_of)
    return run_dir / 'checkpoints' / 'latest.ckpt'

ckpt_path = (pathlib.Path(CKPT) if CKPT else find_best_ckpt(run_dir)).resolve()
out_dir = (pathlib.Path(OUT_DIR) if OUT_DIR else run_dir / 'eval').resolve()
out_dir.mkdir(parents=True, exist_ok=True)

print(f'ckpt    : {ckpt_path}')
print(f'out_dir : {out_dir}')

## 2. Load cfg, build workspace, restore weights

Loads the hydra config saved with the run, instantiates the workspace skeleton, loads the checkpoint payload (skipping optimizer state), and selects EMA weights if available.

In [ ]:
cfg = OmegaConf.load(run_dir / '.hydra' / 'config.yaml')
WorkspaceCls = hydra.utils.get_class(cfg._target_)
workspace = WorkspaceCls(cfg)

payload = torch.load(
    open(ckpt_path, 'rb'), pickle_module=dill,
    map_location='cpu', weights_only=False,
)
workspace.load_payload(payload, exclude_keys=['optimizer'])

policy = workspace.model
if cfg.training.use_ema and workspace.ema_model is not None:
    policy = workspace.ema_model
    print('using EMA weights')
policy = policy.to(DEVICE).eval()
if NUM_INFERENCE_STEPS is not None:
    policy.num_inference_steps = NUM_INFERENCE_STEPS
    print(f'num_inference_steps overridden -> {NUM_INFERENCE_STEPS}')

## 3. Reproduce the val split and pick an episode

Re-opens the zarr the model was trained on, regenerates the val mask using the same `val_ratio` and `seed`, and resolves the requested `VAL_EPISODE` to absolute frame indices.

In [ ]:
zarr_path = pathlib.Path(cfg.task.dataset.dataset_path) / 'replay_buffer.zarr'
if not zarr_path.is_absolute():
    zarr_path = PROJECT_ROOT / zarr_path
assert zarr_path.exists(), f'zarr not found: {zarr_path}'

obs_keys = list(cfg.task.shape_meta.obs.keys())
rb = ReplayBuffer.copy_from_path(str(zarr_path), keys=obs_keys + ['action'])
val_mask = get_val_mask(
    n_episodes=rb.n_episodes,
    val_ratio=cfg.task.dataset.val_ratio,
    seed=cfg.task.dataset.seed,
)
val_ep_indices = np.flatnonzero(val_mask)
assert VAL_EPISODE < len(val_ep_indices), (
    f'VAL_EPISODE {VAL_EPISODE} out of range (n_val={len(val_ep_indices)})')

ep = int(val_ep_indices[VAL_EPISODE])
ep_ends = np.asarray(rb.episode_ends)
ep_starts = np.concatenate([[0], ep_ends[:-1]])
start, end = int(ep_starts[ep]), int(ep_ends[ep])
T = end - start
print(f'val episodes : {len(val_ep_indices)}')
print(f'episode {ep}  (val idx {VAL_EPISODE}, length {T})')

## 4. Sliding-window inference (receding horizon)

Pulls raw arrays for the episode, then runs `policy.predict_action` over a sliding window with stride `n_action_steps`. Each call's first `n_action_steps` predicted actions are written into the trajectory.

In [ ]:
def build_obs_window(raw_obs: dict[str, np.ndarray], cfg, t: int,
                     n_obs_steps: int, device: str) -> dict[str, torch.Tensor]:
    """Match the formatting done by RealImageTactileDataset.__getitem__."""
    obs_dict = {}
    for k, attr in cfg.task.shape_meta.obs.items():
        window = raw_obs[k][t:t + n_obs_steps]
        if attr.type == 'rgb':
            window = np.moveaxis(window, -1, -3).astype(np.float32) / 255.0
        else:
            width = attr.shape[0]
            window = window[:, :width].astype(np.float32)
        obs_dict[k] = torch.from_numpy(window[None]).to(device)
    return obs_dict

gt_action = np.asarray(rb['action'])[start:end]
raw_obs = {k: np.asarray(rb[k])[start:end] for k in obs_keys}

n_obs_steps = cfg.n_obs_steps
n_action_steps = cfg.n_action_steps
pred_action = np.full_like(gt_action, np.nan)

t = 0
while t + n_obs_steps <= T:
    obs_dict = build_obs_window(raw_obs, cfg, t, n_obs_steps, DEVICE)
    with torch.no_grad():
        result = policy.predict_action(obs_dict)
    chunk = result['action'][0].cpu().numpy()

    out_start = t + n_obs_steps - 1
    out_end = min(out_start + n_action_steps, T)
    usable = out_end - out_start
    pred_action[out_start:out_end] = chunk[:usable]
    t += n_action_steps

valid = ~np.isnan(pred_action).any(axis=1)
print(f'covered frames: {int(valid.sum())} / {T}')

## 5. Per-dim action MSE

In [ ]:
mse = ((pred_action[valid] - gt_action[valid]) ** 2).mean(axis=0)
overall = float(mse.mean())
print('per-dim action MSE:')
for d, m in enumerate(mse):
    print(f'  dim {d}: {m:.5f}')
print(f'overall MSE: {overall:.5f}  (over {int(valid.sum())}/{T} frames)')

## 6. Plot predicted vs. ground truth

Also writes the figure to `<out_dir>/pred_vs_gt_val_ep<VAL_EPISODE>.png`.

In [ ]:
n_dim = gt_action.shape[1]
fig, axes = plt.subplots(n_dim, 1, figsize=(11, 1.4 * n_dim),
                         sharex=True, squeeze=False)
axes = axes.ravel()
for d in range(n_dim):
    axes[d].plot(gt_action[:, d], label='ground truth', linewidth=1.0)
    axes[d].plot(pred_action[:, d], label='predicted',
                 linewidth=1.0, linestyle='--')
    axes[d].set_title(f'action[{d}]  MSE={mse[d]:.5f}', fontsize=9)
    axes[d].grid(alpha=0.3)
    axes[d].legend(fontsize=7, loc='upper right')
axes[-1].set_xlabel('frame within episode')
fig.suptitle(
    f'{run_dir.name}\n'
    f'val episode {ep} (val idx {VAL_EPISODE}) \u2014 overall MSE = {overall:.5f}'
)
fig.tight_layout()

out_path = out_dir / f'pred_vs_gt_val_ep{VAL_EPISODE}.png'
fig.savefig(out_path, dpi=120, bbox_inches='tight')
print(f'wrote {out_path}')
plt.show()